# SNN Training Walkthrough

End-to-end guide for the EDABK neuromorphic gesture-recognition SoC.
Covers the full 6-phase pipeline from raw DVS128 events to a hardware-verified
binary-weight checkpoint.

**What this chip does:** classifies 12 DVS128 hand gestures using a 3-layer
binary-weight Leaky-Integrate-and-Fire (LIF) network implemented in silicon.
The ReRAM crossbar performs Vector-Matrix Multiplication (VMM) in-memory;
digital LIF neurons integrate and threshold the result.  
Achieved **53% hardware / 56% software accuracy** on a real GDS tape-out.

**Key design choice — Flattened Time:**  
This SNN does *not* use spike trains over time. All 256 input axons encode one
complete gesture as spatial channels (`spatial_focus` encoding). The chip evaluates
one gesture in a single pass — it behaves like a 1-bit ANN despite using SNN vocabulary.

---

## Pipeline Overview

| Phase | What happens | Script |
|-------|-------------|--------|
| 1 | DVS128 events → 256-axon uint16 feature vectors | `dvs128_preprocess.py` |
| 2 | Define 3-layer binary-weight LIF model | `snn_model.py` |
| 3 | Train with surrogate-gradient STE | `train_dvs128.py` |
| 4 | Binarise weights → `connection_XXX.txt` files | `export_weights.py` |
| 5 | Run Python bit-accurate reference model | `validate_reference_model.py` |
| 6 | RTL co-simulation via cocotb | `verilog/tb/snn_gesture/` |

## ⚠️ Important Note: Two Parallel RTL Codebases

This project has **two separate Verilog implementations**. Always confirm which one you're working with:

| | Simulation baseline | Tapeout-ready |
|---|---|---|
| **Path** | `verilog/tb/hdl/` | `verilog/rtl/SNN_gesture/hdl/` |
| **Use** | Development & early debugging | Final hardware synthesis |
| **Hardware** | Behavioral (no actual IP) | Shim for real Neuromorphic X1 ReRAM IP |
| **Neurons** | Static accumulator | LIF with leak, threshold, reset |

**These are NOT variants — they are separate designs.** Weight-export scripts and this notebook are compatible with the **tapeout-ready** design in `verilog/rtl/SNN_gesture/hdl/`.

## Setup

Install dependencies. The project uses a dedicated virtualenv (`training/venv/`) but any
Python ≥ 3.10 environment with the packages below will work.

In [ ]:
# Install if not already present
# !pip install torch snntorch tonic numpy

import numpy as np
import torch
import torch.nn as nn

print(f"torch  : {torch.__version__}")
print(f"numpy  : {np.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device : {device}")

### Hardware constants

These mirror `nvm_parameter.py` and `nvm_parameter.vh` exactly — the single source of
truth for all network dimensions.  If you change a value here you must also update both
parameter files before synthesising.

In [ ]:
# ── Hardware parameters (sync with nvm_parameter.py / nvm_parameter.vh) ──────
NUM_AXON           = 256   # physical tile width (DVS128 uses 242, zero-padded)
NUM_NEURON         = 64    # neurons per ReRAM crossbar tile
NUM_CLASS          = 12    # gesture classes (DVS128 dataset)

# Layer topology: 13-4-4 cores
NUM_CORES_LAYER_0  = 13   # L0: 256 axons → 13 × 64 = 832 spikes (broadcast)
NUM_CORES_LAYER_1  = 4    # L1: 832 spikes →  4 × 64 = 256 spikes (partitioned)
NUM_CORES_LAYER_2  = 4    # L2: 256 spikes →  4 × 60 = 240 output (broadcast)

NUM_NEURONS_LAYER_0 = NUM_CORES_LAYER_0 * NUM_NEURON   # 832
NUM_NEURONS_LAYER_1 = NUM_CORES_LAYER_1 * NUM_NEURON   # 256
NUM_NEURONS_LAYER_2 = NUM_CORES_LAYER_2 * NUM_NEURON   # 256

# L2 output voting: 240 neurons → 12 classes via interleaved i%12 mapping
VOTES_PER_CLASS    = 20
NUM_VOTES          = NUM_CLASS * VOTES_PER_CLASS        # 240

NEURON_THRESHOLD   = 4     # LIF threshold for spike generation (learned during training)
NEURON_LEAK_SHIFT  = 16    # leak = potential >> NEURON_LEAK_SHIFT  (applied once at picture_done)

print(f"L0: {NUM_AXON} axons  →  {NUM_CORES_LAYER_0} cores × {NUM_NEURON} = {NUM_NEURONS_LAYER_0} spikes")
print(f"L1: {NUM_NEURONS_LAYER_0} spikes →  {NUM_CORES_LAYER_1} cores × {NUM_NEURON} = {NUM_NEURONS_LAYER_1} spikes (partitioned)")
print(f"L2: {NUM_NEURONS_LAYER_1} spikes →  {NUM_CORES_LAYER_2} cores × {VOTES_PER_CLASS} = {NUM_VOTES} output (broadcast, trimmed)")
print(f"Voting: {NUM_VOTES} neurons → {NUM_CLASS} classes  ({VOTES_PER_CLASS} votes / class)")

---

## Phase 1 — DVS128 Preprocessing

### Why spatial_focus?

DVS128 is a 128×128 event camera that fires per pixel when brightness changes.
We need to compress the stream into exactly 256 axons for the hardware tile.

We use **`spatial_focus`** encoding:
- Split events at the midpoint timestamp → 2 temporal windows.
- For each window: sum-pool to an 11×11 spatial grid (merged ON+OFF polarity).
- Concatenate → 242 features. Zero-pad to 256 to fill the tile.

The 11×11 grid captures finer spatial detail than the 8×8 `temporal2` default,
at the cost of 14 zero-padded axons.

**Normalisation:** clip at 99th-percentile, scale to `[0, 32767]` (uint16).
This prevents hardware 16-bit accumulator overflow during inference.

In [ ]:
# ── Inlined from dvs128_preprocess.py ────────────────────────────────────────

SENSOR_H  = 128
SENSOR_W  = 128
MAX_UINT15 = 32767   # clips to 15-bit range to avoid signed overflow in HW MAC

def _sum_pool_2d(arr: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    """Proportional sum-pool a 2-D array to (out_h, out_w)."""
    H, W = arr.shape
    pooled = np.zeros((out_h, out_w), dtype=np.float64)
    for i in range(out_h):
        h0 = round(i       * H / out_h)
        h1 = round((i + 1) * H / out_h)
        for j in range(out_w):
            w0 = round(j       * W / out_w)
            w1 = round((j + 1) * W / out_w)
            pooled[i, j] = arr[h0:h1, w0:w1].sum()
    return pooled


def events_to_features_spatial_focus(events: np.ndarray) -> np.ndarray:
    """
    spatial_focus: 2 windows × 11×11 merged polarity = 242 axons, padded to 256.
    events: structured array with fields 't', 'x', 'y', 'p'.
    """
    t = events['t']
    t_mid = (t.min() + t.max()) / 2.0
    windows = []
    for mask in [t <= t_mid, t > t_mid]:
        hist = np.zeros((SENSOR_H, SENSOR_W), dtype=np.float64)
        ev = events[mask]
        if len(ev):
            np.add.at(hist, (ev['y'].astype(np.intp), ev['x'].astype(np.intp)), 1.0)
        pooled = _sum_pool_2d(hist, 11, 11)
        windows.append(pooled.ravel())
    vec = np.concatenate(windows)            # (242,)
    return np.pad(vec, (0, 14), mode='constant')  # pad to (256,)


def normalise(features_raw: np.ndarray, clip_val: float = None):
    """Clip at 99th-percentile, scale to [0, 32767] uint16."""
    if clip_val is None:
        clip_val = float(np.percentile(features_raw, 99))
    clipped = np.clip(features_raw, 0.0, clip_val)
    scaled  = np.round(clipped / clip_val * MAX_UINT15).astype(np.uint16)
    return scaled, clip_val


# ── Demonstrate on a synthetic event stream ───────────────────────────────────
rng = np.random.default_rng(42)
N_EVENTS = 5000
fake_events = np.zeros(N_EVENTS, dtype=[
    ('t', np.int64), ('x', np.uint8), ('y', np.uint8), ('p', np.uint8)
])
fake_events['t'] = np.sort(rng.integers(0, 1_000_000, N_EVENTS))
fake_events['x'] = rng.integers(0, 128, N_EVENTS)
fake_events['y'] = rng.integers(0, 128, N_EVENTS)
fake_events['p'] = rng.integers(0, 2, N_EVENTS)

feat_raw = events_to_features_spatial_focus(fake_events)
feat_norm, clip_val = normalise(feat_raw[None], None)   # add batch dim
print(f"Raw feature shape : {feat_raw.shape}  (242 real + 14 zero-padded = 256)")
print(f"Normalised dtype  : {feat_norm.dtype}")
print(f"Normalised range  : [{int(feat_norm.min())}, {int(feat_norm.max())}]  (expected [0, 32767])")
print(f"Pad verification  : axons 242-255 = {feat_norm[0, 242:].tolist()}  (should be all 0)")

### Run real preprocessing

Skip this cell if you already have `data/dvs128_train.npz` and `data/dvs128_test.npz`.
The first run downloads ~1.5 GB from figshare.

In [ ]:
# Uncomment to preprocess from scratch:
# import subprocess, sys
# subprocess.run([sys.executable, "dvs128_preprocess.py",
#                 "--encoding", "spatial_focus"], check=True)

# Load already-preprocessed data:
import pathlib
train_path = pathlib.Path("data/dvs128_train.npz")
test_path  = pathlib.Path("data/dvs128_test.npz")

if train_path.exists() and test_path.exists():
    train_data = np.load(train_path)
    test_data  = np.load(test_path)
    X_train = train_data["features"].astype(np.float32)
    y_train = train_data["labels"]
    X_test  = test_data["features"].astype(np.float32)
    y_test  = test_data["labels"]
    print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
    print(f"Classes: {sorted(np.unique(y_train).tolist())}")
else:
    print("data/*.npz not found — using small synthetic dataset for demonstration.")
    N = 200
    X_train = torch.randint(0, 256, (N, 256)).float().numpy()
    y_train = np.tile(np.arange(NUM_CLASS), N // NUM_CLASS + 1)[:N]
    X_test  = torch.randint(0, 256, (40, 256)).float().numpy()
    y_test  = np.tile(np.arange(NUM_CLASS), 40 // NUM_CLASS + 1)[:40]
    print(f"Synthetic: train {X_train.shape}  |  test {X_test.shape}")

---

## Phase 2 — Model Architecture

### Why binary weights?

The ReRAM crossbar stores one bit per synapse: programmed (HRS) = 0, erased (LRS) = 1.
Weights are physically constrained to {0, 1}. The odd-axon sign convention (odd inputs
negated before the MAC) gives the network the equivalent of {+1, -1} weights.

### Why a single time step?

The chip evaluates each gesture in a single Wishbone pass.  Training matches this:
membrane potential is reset to 0 before each sample, and the LIF is called exactly once.

### Architecture diagram

```
Input (256 axons, uint16)
        │  odd axons negated (axon_sign buffer)
        ▼
Layer 0 │  13 cores × 64 neurons  (broadcast — every core sees all 256 axons)
        │  BinaryLinear + LIF   →  832 spikes
        ▼
Layer 1 │   4 cores × 64 neurons  (partitioned — each core sees 208 spikes)
        │  BinaryLinear + LIF   →  256 spikes
        ▼
Layer 2 │   4 cores × 60 neurons  (broadcast — each core sees all 256 spikes)
        │  BinaryLinear + LIF   →  240 output spikes
        ▼
Vote    │  interleaved: spike[i] votes for class (i % 12)
        │  sum 20 votes per class → 12 logits → argmax
        ▼
Prediction (0–11)
```

In [ ]:
---

## Phase 3 — Training

### Critical Design: Single Time Step ("Flattened Time")

**Unlike traditional SNNs**, this network does NOT use spike trains over time. All 256 input axons encode a complete gesture as spatial channels in a **single snapshot**. The chip evaluates one gesture in one forward pass.

> **Important**: Leak is applied **once per gesture** at `picture_done`, NOT per-axon during accumulation.  
> Applying leak per-axon inside the sequential accumulation loop creates severe input-order bias: early axons decay ~256× more than late axons, destroying accuracy. This was Root Cause F-24 (−30.5pp impact). Solution: accumulate all axons first, apply leak+threshold once at the end.

### Loss function

- **Primary:** Cross-entropy on interleaved votes (spike[i] → class i%12).
- **Spike-rate regularisation (λ_reg):** penalises layers whose spike rate is outside
  [5%, 50%]. Bootstraps the network out of the dead-neuron problem when thresholds
  start too high.
- **L2 weight regularisation:** keeps continuous weights near the binarisation boundary,
  preventing them from collapsing to all-0 or all-1.

### 4-phase schedule

| Phase | Epochs | LR | Surrogate slope | Purpose |
|-------|--------|-----|-----------------|--------|
| 1 | 1–10  | 1e-3 | 0.6→0.6  | Establish healthy spike rates |
| 2 | 11–30 | 5e-4 | 0.6→3.0  | Adapt to binary quantisation |
| 3 | 31–60 | 1e-4 | 3.0→10.0 | Maximise accuracy |
| 4 | 61–80 | 5e-5 | 10.0     | Fine-tune; freeze thresholds |

---

## Phase 3 — Training

### Loss function

- **Primary:** Cross-entropy on interleaved votes (spike[i] → class i%12).
- **Spike-rate regularisation (λ_reg):** penalises layers whose spike rate is outside
  [5%, 50%]. Bootstraps the network out of the dead-neuron problem when thresholds
  start too high.
- **L2 weight regularisation:** keeps continuous weights near the binarisation boundary,
  preventing them from collapsing to all-0 or all-1.

### 4-phase schedule

| Phase | Epochs | LR | Surrogate slope | Purpose |
|-------|--------|-----|-----------------|--------|
| 1 | 1–10  | 1e-3 | 0.6→0.6  | Establish healthy spike rates |
| 2 | 11–30 | 5e-4 | 0.6→3.0  | Adapt to binary quantisation |
| 3 | 31–60 | 1e-4 | 3.0→10.0 | Maximise accuracy |
| 4 | 61–80 | 5e-5 | 10.0     | Fine-tune; freeze thresholds |

In [ ]:
# ── Inlined training loop (simplified; full logic in train_dvs128.py) ─────────
import torch.nn.functional as F

SPIKE_TARGET = 0.28   # target spike rate for all layers

def spike_rate_loss(spikes, target=SPIKE_TARGET, lam=0.05):
    """Asymmetric penalty: firing below target hurts 10× more than above."""
    rate = spikes.mean()
    if rate < target:
        return lam * 10.0 * (target - rate) ** 2
    return lam * (rate - target) ** 2


def l2_reg(modules, lam=1e-3):
    """L2 on continuous weight parameters only (not LIF threshold/beta)."""
    loss = torch.tensor(0.0)
    for m in modules:
        for core in m:
            loss = loss + core.linear.weight.pow(2).sum()
    return lam * loss


def train_epoch(l0, l1, l2, optimizer, X, y, batch_size=32, lam_reg=0.05):
    """One training epoch. Returns mean loss and accuracy."""
    X_t = torch.tensor(X, device=device)
    y_t = torch.tensor(y, dtype=torch.long, device=device)
    idx = torch.randperm(len(X_t))
    total_loss, total_correct, total_n = 0.0, 0, 0

    for start in range(0, len(X_t), batch_size):
        batch_x = X_t[idx[start:start + batch_size]]
        batch_y = y_t[idx[start:start + batch_size]]

        out0, out1, out2 = forward_network(l0, l1, l2, batch_x)
        logits = interleaved_logits(out2)

        loss = (F.cross_entropy(logits, batch_y)
                + spike_rate_loss(out0, lam=lam_reg)
                + spike_rate_loss(out1, lam=lam_reg)
                + spike_rate_loss(out2, lam=lam_reg)
                + l2_reg([l0, l1, l2]))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss    += loss.item() * len(batch_x)
        total_correct += (logits.argmax(1) == batch_y).sum().item()
        total_n       += len(batch_x)

    return total_loss / total_n, total_correct / total_n


def evaluate(l0, l1, l2, X, y, batch_size=64):
    """Return accuracy on the given split."""
    X_t = torch.tensor(X, device=device)
    y_t = torch.tensor(y, dtype=torch.long, device=device)
    correct, total = 0, 0
    with torch.no_grad():
        for start in range(0, len(X_t), batch_size):
            bx = X_t[start:start + batch_size]
            by = y_t[start:start + batch_size]
            _, _, out2 = forward_network(l0, l1, l2, bx)
            logits = interleaved_logits(out2)
            correct += (logits.argmax(1) == by).sum().item()
            total   += len(bx)
    return correct / total


# Move all modules to device
for layer in [l0, l1, l2]:
    for core in layer:
        core.to(device)

all_params_list = [p for layer in [l0, l1, l2] for core in layer for p in core.parameters()]
optimizer = torch.optim.Adam(all_params_list, lr=1e-3)

# ── Quick smoke-test: 5 epochs ─────────────────────────────────────────────────
N_EPOCHS = 5
print(f"Training for {N_EPOCHS} epochs (smoke test — run train_dvs128.py for 80 epochs)")
for epoch in range(1, N_EPOCHS + 1):
    loss, train_acc = train_epoch(l0, l1, l2, optimizer, X_train, y_train)
    test_acc        = evaluate(l0, l1, l2, X_test, y_test)
    print(f"Epoch {epoch:2d}  loss={loss:.4f}  train_acc={100*train_acc:.1f}%  test_acc={100*test_acc:.1f}%")

---

## Phase 4 — Weight Export

After training, weights are binarised and written to `connection_XXX.txt` files.
The RTL testbench reads these files to program the ReRAM crossbar.

### File format

Each file represents one 64-neuron crossbar tile (one `LIFCore`):
- **256 lines** (one per axon, zero-padded if n_in < 256)
- **64 characters** per line (`'0'`/`'1'`)
- **MSB-first**: character 0 → neuron 63, character 63 → neuron 0

### Why MSB-first?

The RTL reads weights as a 64-bit word where bit[63] is neuron 0.
Python's `int(binary_str, 2)` treats index 0 as MSB, so character 0 maps to bit 63 = neuron 63.
This matches the hardware memory layout.

In [ ]:
# ── Inlined from export_weights.py ───────────────────────────────────────────
import pathlib

def export_connection_file(W: np.ndarray, filepath: pathlib.Path) -> None:
    """
    Write one crossbar connection file.
    W: (n_in, n_out) binary uint8.
    Physical tile is (NUM_AXON=256, NUM_NEURON=64); extra rows padded with zeros.
    Columns written MSB-first: col[0] of output = rightmost character.
    """
    n_in, n_out = W.shape
    pad_rows = max(0, NUM_AXON  - n_in)
    pad_cols = max(0, NUM_NEURON - n_out)
    W_padded = np.pad(W, ((0, pad_rows), (0, pad_cols)), mode='constant')
    W_trimmed = W_padded[:NUM_AXON, :NUM_NEURON]

    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w') as f:
        for row in range(NUM_AXON):
            # MSB-first: flip column order so neuron 63 is character 0
            bits = ''.join(str(int(b)) for b in reversed(W_trimmed[row]))
            f.write(bits + '\n')


def export_all_weights(l0, l1, l2, out_dir: pathlib.Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    file_idx = 0
    for layer_idx, layer in enumerate([l0, l1, l2]):
        is_last = (layer_idx == 2)
        for core_idx, core in enumerate(layer):
            W = core.linear.get_binary_weights()   # (n_in, 64)
            if is_last:
                # Zero out inactive output neurons (beyond VOTES_PER_CLASS)
                W_out = W.copy()
                W_out[:, VOTES_PER_CLASS:] = 0
            else:
                W_out = W
            path = out_dir / f"connection_{file_idx:03d}"
            export_connection_file(W_out, path)
            file_idx += 1
    print(f"Exported {file_idx} connection files to {out_dir}/")


# ── Demo export ───────────────────────────────────────────────────────────────
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = pathlib.Path(tmp) / "connection"
    export_all_weights(l0, l1, l2, tmp_path)
    files = sorted(tmp_path.glob("connection_*"))
    print(f"\nTotal files: {len(files)}  (= 13 L0 + 4 L1 + 4 L2 = 21 cores)")
    # Inspect first file
    with open(files[0]) as f:
        lines = f.readlines()
    print(f"connection_000: {len(lines)} lines × {len(lines[0].strip())} chars")
    print(f"First line (axon 0, all 64 neuron connections, MSB-first): {lines[0].strip()[:20]}...")

---

## Phase 5 — Hardware Validation (Python Reference Model)

Before running the slow cocotb RTL simulation, we use a Python reference model
that applies the **exact same fixed-point arithmetic** as the RTL:

1. Integer accumulation (no floating point)
2. Arithmetic-right-shift leak: `potential >> NEURON_LEAK_SHIFT`
3. Integer threshold comparison
4. Interleaved i%12 voting

If reference-model accuracy matches the PyTorch (software) accuracy, the trained
weights are hardware-compatible.  If it diverges, there is a HW/SW contract
violation that will also appear in RTL simulation — catch it here first.

### Key differences from the PyTorch model

| | PyTorch model | Hardware reference |
|---|---|---|
| Accumulation | float32 MAC | integer sum over binary axons |
| Leak | beta ≈ 0.999 (learned) | `>> LEAK_SHIFT` (integer) |
| Threshold | float (learned) | integer register value |
| Weights | continuous → binarised at inference | loaded from connection_XXX.txt |

In [ ]:
---

## Phase 6 — RTL Co-simulation

After Phase 5 confirms the weights are hardware-compatible, run the cocotb RTL
testbench to verify the actual Verilog against the Python reference model.

The 7-phase cocotb suite verifies the **tapeout-ready RTL** in `verilog/rtl/SNN_gesture/hdl/`:
- Wishbone interface timing
- ReRAM synapse matrix programming + readback
- LIF neuron block (accumulate, leak, threshold, spike)
- Spike output SRAM
- Single-core end-to-end
- Layer inference
- Full network inference

```bash
# From project root
cd verilog/tb/snn_gesture/cocotb
make MODULE=test_neuron_block SIM=icarus   # Phase 3: LIF neuron block
make MODULE=test_single_core  SIM=icarus   # Phase 5: single core E2E
make MODULE=test_layer_inference SIM=icarus  # Phase 6: layer inference
```

See [`verilog/tb/snn_gesture/README.md`](../verilog/tb/snn_gesture/README.md) for
full test commands and hardware invariants.

---

## Summary

| Step | Command | Output |
|------|---------|--------|
| Preprocess | `python dvs128_preprocess.py --encoding spatial_focus` | `data/dvs128_*.npz` |
| Train | `python train_dvs128.py --epochs 80` | `checkpoints/best.pt` |
| Export | `python export_weights.py --checkpoint checkpoints/best.pt` | `mem/connection/connection_*.txt` |
| Validate | `python validate_reference_model.py --checkpoint checkpoints/best.pt` | HW/SW accuracy report |
| RTL sim | `cd verilog/tb/snn_gesture/cocotb && make` | cocotb pass/fail |

For complete system context, see:
- **Data flow & topology**: [`snn_gesture_working.md`](../snn_gesture_working.md)
- **Training pipeline details**: [`training/README.md`](README.md)
- **RTL reference & verification**: [`verilog/rtl/SNN_gesture/README.md`](../verilog/rtl/SNN_gesture/README.md)

---

## Phase 6 — RTL Co-simulation

After Phase 5 confirms the weights are hardware-compatible, run the cocotb RTL
testbench to verify the actual Verilog against the Python reference model.

```bash
# From project root
cd verilog/tb/snn_gesture/cocotb
make MODULE=test_neuron_block SIM=icarus   # Phase 3: LIF neuron block
make MODULE=test_single_core  SIM=icarus   # Phase 5: single core E2E
make MODULE=test_layer_inference SIM=icarus  # Phase 6: layer inference
```

The 7-phase cocotb suite verifies:
- Wishbone interface timing
- ReRAM synapse matrix programming + readback
- LIF neuron block (accumulate, leak, threshold, spike)
- Spike output SRAM
- Single-core end-to-end
- Layer inference
- Full network inference

See [`verilog/tb/snn_gesture/README.md`](../verilog/tb/snn_gesture/README.md) for
hardware invariants and test commands.

---

## Summary

| Step | Command | Output |
|------|---------|--------|
| Preprocess | `python dvs128_preprocess.py --encoding spatial_focus` | `data/dvs128_*.npz` |
| Train | `python train_dvs128.py --epochs 80` | `checkpoints/best.pt` |
| Export | `python export_weights.py --checkpoint checkpoints/best.pt` | `mem/connection/connection_*.txt` |
| Validate | `python validate_reference_model.py --checkpoint checkpoints/best.pt` | HW/SW accuracy report |
| RTL sim | `cd verilog/tb/snn_gesture/cocotb && make` | cocotb pass/fail |

For the full system context, see [`snn_gesture_working.md`](../snn_gesture_working.md).